In [ ]:
#analyzing congress right now 
import pandas as pd
import plotly.io as pio
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

pio.renderers.default = "browser"

# reading
votes = pd.read_csv('data/HS119/HS119_votes.csv')
members = pd.read_csv('data/HS119/HS119_members.csv')
roll_calls = pd.read_csv('data/HS119   /HS119_rollcalls.csv')

house_votes = votes[votes["chamber"] == "House"]
senate_votes = votes[votes["chamber"] == "Senate"]

house_members = members[members["chamber"] == "House"]
senate_members = members[members["chamber"] == "Senate"]

# mapping
def convert_vote(code):
    if code in [1, 2, 3]:
        return 1;
    elif code in [4, 5, 6]:
        return -1;
    else:
        return 0;

house_votes["votes_numeric"] = votes["cast_code"].apply(convert_vote)
senate_votes["votes_numeric"] = votes["cast_code"].apply(convert_vote)

house_matrix = house_votes.pivot_table(index="icpsr", columns="rollnumber", values="votes_numeric", fill_value=0)
senate_matrix = senate_votes.pivot_table(index="icpsr", columns="rollnumber", values="votes_numeric", fill_value=0)

# Keep only legislators with at least 100 cast votes (non-zero entries).
house_matrix = house_matrix[(house_matrix != 0).sum(axis=1) >= 100]
senate_matrix = senate_matrix[(senate_matrix != 0).sum(axis=1) >= 100]


# L2-normalize each legislator vector (row) before PCA
house_matrix_norm = pd.DataFrame(
    normalize(house_matrix, norm="l2", axis=1),
    index=house_matrix.index,
    columns=house_matrix.columns,
 )
senate_matrix_norm = pd.DataFrame(
    normalize(senate_matrix, norm="l2", axis=1),
    index=senate_matrix.index,
    columns=senate_matrix.columns,
 )

# PCA

pca = PCA(n_components=2)
coords_house = pca.fit_transform(house_matrix_norm)
coords_senate = pca.fit_transform(senate_matrix_norm)

# plot house
df_house = pd.DataFrame(coords_house, columns=["PC1", "PC2"])
df_house["icpsr"] = house_matrix_norm.index

df_house = df_house.merge(
    house_members[["icpsr", "party_code", "bioname"]],
    on="icpsr"
)


fig = px.scatter(
    df_house,
    title="House",
    x="PC1",
    y="PC2",
    color="party_code",
    hover_name="bioname"
)

fig.show()

# plot senate
df_senate = pd.DataFrame(coords_senate, columns=["PC1", "PC2"])
df_senate["icpsr"] = senate_matrix_norm.index

df_senate = df_senate.merge(
    senate_members[["icpsr", "party_code", "bioname"]],
    on="icpsr"
)


fig = px.scatter(
    df_senate,
    title="Senate",
    x="PC1",
    y="PC2",
    color="party_code",
    hover_name="bioname"
)

fig.show()

In [1]:
# all congress polarization
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import subprocess
from pathlib import Path

# reading
votes = pd.read_csv('data/all/HSall_votes.csv')
members = pd.read_csv('data/all/HSall_members.csv')

def convert_vote(code):
    if code in [1, 2, 3]:
        return 1
    elif code in [4, 5, 6]:
        return -1
    elif code in [7, 8]:
        return 0
    else:
        return np.nan
    
votes["vote_numeric"] = votes["cast_code"].apply(convert_vote)

polarization_results = []

for congress in sorted(votes["congress"].unique()):
    v = votes[votes["congress"] == congress]

    matrix = v.pivot_table(
        index="icpsr",
        columns="rollnumber",
        values="vote_numeric"
    )

    # Require at least 100 votes
    vote_counts = matrix.notna().sum(axis=1)
    matrix = matrix[vote_counts >= 100]

    if matrix.shape[0] < 10 or matrix.shape[1] < 5:
        continue

    M = matrix.fillna(0).values

    # Row normalize
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    M_norm = M / np.where(norms == 0, 1, norms)

    pca = PCA(n_components=3)
    coords = pca.fit_transform(M_norm)

    pc1_variance = np.var(coords[:, 0])
    explained_pc1 = pca.explained_variance_ratio_[0]

    polarization_results.append({
        "congress": congress,
        "pc1_variance_polarization": pc1_variance,
        "pc1_explained_variance": explained_pc1,
        "num_legislators": matrix.shape[0],
        "num_votes": matrix.shape[1]
    })

polarization_df = pd.DataFrame(polarization_results)
print(polarization_df.head())

import plotly.express as px

fig = px.line(
    polarization_df,
    x="congress",
    y="pc1_variance_polarization",
    markers=True,
    title="Party-Independent Polarization Over Time",
    labels={
        "congress": "Congress",
        "pc1_variance_polarization": "PC1 Variance"
    }
)

output_path = Path("polarization.html")
fig.write_html(output_path, include_plotlyjs="cdn", auto_open=False)
subprocess.Popen(["open", "-a", "Google Chrome", str(output_path)])




   congress  pc1_variance_polarization  pc1_explained_variance  \
0         1                   0.243826                0.277607   
1         5                   0.422047                0.448816   
2         6                   0.530714                0.608192   
3         7                   0.448568                0.523501   
4         8                   0.252008                0.283633   

   num_legislators  num_votes  
0               30        109  
1              114        202  
2               19        120  
3               73        142  
4              110        150  


<Popen: returncode: None args: ['open', '-a', 'Google Chrome', 'polarization...>

In [2]:
# modern polarization, starting in the 1970s. difference between party centroids 
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

MODERN_START = 93   # change this to 91, 92, 93, etc.
MIN_VOTES = 100

def convert_vote(code):
    if code in [1, 2, 3]:
        return 1
    elif code in [4, 5, 6]:
        return -1
    elif code in [7, 8]:
        return 0
    else:
        return np.nan

votes["vote_numeric"] = votes["cast_code"].apply(convert_vote)

modern_results = []

for congress in sorted(votes["congress"].unique()):
    if congress < MODERN_START:
        continue

    v = votes[votes["congress"] == congress]

    matrix = v.pivot_table(
        index="icpsr",
        columns="rollnumber",
        values="vote_numeric"
    )

    vote_counts = matrix.notna().sum(axis=1)
    matrix = matrix[vote_counts >= MIN_VOTES]

    if matrix.shape[0] < 10 or matrix.shape[1] < 5:
        continue

    M = matrix.fillna(0).values

    norms = np.linalg.norm(M, axis=1, keepdims=True)
    M_norm = M / np.where(norms == 0, 1, norms)

    pca = PCA(n_components=2)
    coords = pca.fit_transform(M_norm)

    df = pd.DataFrame(coords, columns=["PC1", "PC2"])
    df["icpsr"] = matrix.index

    df = df.merge(
        members[["icpsr", "party_code", "bioname"]],
        on="icpsr",
        how="left"
    )

    # Voteview: 100 = Democrat, 200 = Republican
    df = df[df["party_code"].isin([100, 200])]

    if df["party_code"].nunique() < 2:
        continue

    dem_center = df[df["party_code"] == 100][["PC1", "PC2"]].mean()
    rep_center = df[df["party_code"] == 200][["PC1", "PC2"]].mean()

    distance = np.linalg.norm(rep_center - dem_center)

    modern_results.append({
        "congress": congress,
        "party_centroid_distance": distance,
        "dem_pc1_mean": dem_center["PC1"],
        "rep_pc1_mean": rep_center["PC1"],
        "dem_pc2_mean": dem_center["PC2"],
        "rep_pc2_mean": rep_center["PC2"],
        "num_legislators": len(df),
        "num_votes": matrix.shape[1]
    })

modern_polarization = pd.DataFrame(modern_results)
print(modern_polarization.head())

import plotly.express as px

fig = px.line(
    modern_polarization,
    x="congress",
    y="party_centroid_distance",
    markers=True,
    title=f"Democrat-Republican Polarization Since Congress {MODERN_START}",
    labels={
        "congress": "Congress",
        "party_centroid_distance": "Distance Between Party Centers"
    }
)

fig.show()

   congress  party_centroid_distance  dem_pc1_mean  rep_pc1_mean  \
0        93                 0.458910     -0.197581      0.258244   
1        94                 0.536054     -0.175470      0.358367   
2        95                 0.512466     -0.177585      0.332187   
3        96                 0.612728     -0.243457      0.369038   
4        97                 0.571110     -0.275686      0.285594   

   dem_pc2_mean  rep_pc2_mean  num_legislators  num_votes  
0      0.047306     -0.005816             5469       1138  
1      0.033744     -0.014965             5541       1311  
2      0.038648     -0.013829             5492       1540  
3      0.034883      0.017985             5493       1276  
4     -0.024321      0.081184             5495        966  


In [1]:
# member ideology trajectories by congress
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

DATA_DIR = Path("data/all")
OUTPUT_DIR = Path("data/derived")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_CHAMBERS = {"House", "Senate"}
DEMOCRAT_CODES = {100, 328}
REPUBLICAN_CODES = {200}

def convert_vote(code):
    if code in (1, 2, 3):
        return 1
    if code in (4, 5, 6):
        return -1
    return 0

def build_year_labels(rollcalls):
    labels = {}
    for congress, group in rollcalls.groupby("congress"):
        dates = pd.to_datetime(group["date"], errors="coerce").dropna()
        if dates.empty:
            labels[int(congress)] = str(int(congress))
            continue
        start_year = int(dates.min().year)
        end_year = int(dates.max().year)
        labels[int(congress)] = str(start_year) if start_year == end_year else f"{start_year}-{end_year}"
    return labels

def align_ideology_axis(df):
    aligned = df.copy()
    dem_mask = aligned["party_code"].isin(DEMOCRAT_CODES)
    rep_mask = aligned["party_code"].isin(REPUBLICAN_CODES)

    if dem_mask.any() and rep_mask.any():
        dem_mean = aligned.loc[dem_mask, "pc1"].mean()
        rep_mean = aligned.loc[rep_mask, "pc1"].mean()
        if rep_mean < dem_mean:
            aligned["pc1"] *= -1
            aligned["pc2"] *= -1

    ideology_std = aligned["pc1"].std(ddof=0)
    ideology_std = float(ideology_std) if pd.notna(ideology_std) and ideology_std > 0 else 1.0
    ideology_mean = float(aligned["pc1"].mean())
    aligned["display_ideology_score"] = (aligned["pc1"] - ideology_mean) / ideology_std

    dem_mask = aligned["party_code"].isin(DEMOCRAT_CODES)
    rep_mask = aligned["party_code"].isin(REPUBLICAN_CODES)
    if dem_mask.any() and rep_mask.any():
        dem_mean = float(aligned.loc[dem_mask, "pc1"].mean())
        rep_mean = float(aligned.loc[rep_mask, "pc1"].mean())
        gap = rep_mean - dem_mean
        midpoint = (rep_mean + dem_mean) / 2.0
        aligned["party_gap_score"] = (aligned["pc1"] - midpoint) / gap if abs(gap) > 1e-12 else np.nan
    else:
        aligned["party_gap_score"] = np.nan

    return aligned

def compute_member_ideology(votes, members, year_labels, chamber, min_votes=100, min_legislators=10, min_rollcalls=5):
    chamber_votes = votes[votes["chamber"] == chamber].copy()
    chamber_members = members[members["chamber"] == chamber].copy()
    chamber_votes["vote_numeric"] = chamber_votes["cast_code"].apply(convert_vote)

    result_frames = []
    meta_rows = []

    for congress, congress_votes in chamber_votes.groupby("congress", sort=True):
        congress_members = chamber_members[chamber_members["congress"] == congress]
        if congress_members.empty:
            continue

        matrix = congress_votes.pivot_table(index="icpsr", columns="rollnumber", values="vote_numeric", fill_value=0)
        if matrix.empty:
            continue

        vote_counts = (matrix != 0).sum(axis=1)
        matrix = matrix[vote_counts >= min_votes]
        if matrix.shape[0] < min_legislators or matrix.shape[1] < min_rollcalls:
            continue

        coords = PCA(n_components=2).fit_transform(normalize(matrix, norm="l2", axis=1))
        ideology_df = pd.DataFrame({
            "icpsr": matrix.index.astype(int),
            "pc1": coords[:, 0],
            "pc2": coords[:, 1],
            "cast_vote_count": vote_counts.loc[matrix.index].to_numpy(),
        })

        ideology_df = ideology_df.merge(
            congress_members[["icpsr", "party_code", "bioname", "state_abbrev", "district_code", "bioguide_id"]],
            on="icpsr",
            how="left",
        )
        ideology_df = align_ideology_axis(ideology_df)
        ideology_df["congress"] = int(congress)
        ideology_df["year_label"] = year_labels.get(int(congress), str(int(congress)))
        ideology_df["chamber"] = chamber
        result_frames.append(ideology_df)

        meta_rows.append({
            "congress": int(congress),
            "year_label": year_labels.get(int(congress), str(int(congress))),
            "chamber": chamber,
            "num_legislators": int(matrix.shape[0]),
            "num_rollcalls": int(matrix.shape[1]),
        })

    return pd.concat(result_frames, ignore_index=True), pd.DataFrame(meta_rows)

def build_member_summary(records):
    summary_rows = []
    for icpsr, group in records.groupby("icpsr"):
        ordered = group.sort_values(["congress", "chamber"]).reset_index(drop=True)
        scores = ordered["display_ideology_score"].to_numpy()
        deltas = np.diff(scores) if len(scores) > 1 else np.array([])
        summary_rows.append({
            "icpsr": int(icpsr),
            "bioname": ordered.loc[0, "bioname"],
            "appearances": int(len(ordered)),
            "first_congress": int(ordered.loc[0, "congress"]),
            "last_congress": int(ordered.loc[len(ordered) - 1, "congress"]),
            "net_shift": float(scores[-1] - scores[0]),
            "cumulative_shift": float(np.abs(deltas).sum()) if len(deltas) else 0.0,
        })
    return pd.DataFrame(summary_rows).sort_values(["appearances", "last_congress", "bioname"], ascending=[False, False, True])

def write_json(path, rows):
    path.write_text(json.dumps(rows, indent=2), encoding="utf-8")

votes = pd.read_csv(DATA_DIR / "HSall_votes.csv", usecols=["congress", "chamber", "rollnumber", "icpsr", "cast_code"], low_memory=False)
members = pd.read_csv(DATA_DIR / "HSall_members.csv", usecols=["congress", "chamber", "icpsr", "party_code", "bioname", "state_abbrev", "district_code", "bioguide_id"], low_memory=False)
rollcalls = pd.read_csv(DATA_DIR / "HSall_rollcalls.csv", usecols=["congress", "date"], low_memory=False)

members = members[members["chamber"].isin(VALID_CHAMBERS)].copy()
year_labels = build_year_labels(rollcalls)

ideology_frames = []
meta_frames = []
for chamber in ["House", "Senate"]:
    records, meta = compute_member_ideology(votes, members, year_labels, chamber)
    ideology_frames.append(records)
    meta_frames.append(meta)

ideology_records = pd.concat(ideology_frames, ignore_index=True).sort_values(["bioname", "congress", "chamber"])
congress_meta = pd.concat(meta_frames, ignore_index=True).sort_values(["congress", "chamber"])
summary = build_member_summary(ideology_records)

ideology_records.to_csv(OUTPUT_DIR / "member_ideology_records.csv", index=False)
summary.to_csv(OUTPUT_DIR / "member_ideology_summary.csv", index=False)
congress_meta.to_csv(OUTPUT_DIR / "congress_ideology_metadata.csv", index=False)
write_json(OUTPUT_DIR / "member_ideology_records.json", ideology_records.replace({np.nan: None}).to_dict(orient="records"))
write_json(OUTPUT_DIR / "member_ideology_summary.json", summary.replace({np.nan: None}).to_dict(orient="records"))

summary[["bioname", "appearances", "first_congress", "last_congress", "net_shift"]].head(20)


,bioname,appearances,first_congress,last_congress,net_shift
2207,"DINGELL, John David, Jr.",29,85,113,0.131700
1161,"BYRD, Robert Carlyle",29,83,111,0.899327
3566,"HAYDEN, Carl Trumbull",28,62,90,0.228954
8894,"CONYERS, John, Jr.",27,89,115,0.028835
4038,"INOUYE, Daniel Ken",27,86,112,-0.190791
9310,"GRASSLEY, Charles Ernest",26,94,119,-0.244699
9420,"MARKEY, Edward John",26,95,119,0.063692
8410,"WHITTEN, Jamie Lloyd",26,78,103,-0.146530
9263,"YOUNG, Donald Edwin",25,93,117,-0.270292
9640,"WYDEN, Ronald Lee",24,97,119,0.155718
